# 08_production_serving_and_profiling: Serving Endpoint Client Simulation

This notebook profiles production serving endpoints. We implement an asynchronous streaming SSE client to measure Time-To-First-Token (TTFT), Time-Per-Output-Token (TPOT/ITL), and throughput (Tokens-Per-Second) metrics.

### Core Engineering Intuitions
- **TTFT (Time-To-First-Token)**: Measures user-responsiveness. Highly dependent on network overhead and prompt length prefill computations.
- **TPOT / ITL (Time-Per-Output-Token)**: The average interval between successive generated tokens. Must match reading speed comfort ($15-25$ ms per token).
- **Throughput (TPS)**: Raw system pipeline throughput. Scales with batch size, but larger batches increase queue delays, degrading TTFT and TPOT. Balancing these constraints is the core challenge of serving gateway SLAs.

### Batch Scaling SLA Trade-offs
- **Small Batch Size**: Yields low TTFT and low TPOT (extremely fast response for the active user), but wastes GPU compute cycles (low throughput).
- **Large Batch Size**: Optimizes GPU compute saturation and TPS (max tokens/sec), but increases queue wait times, blowing up client TTFT and TPOT.

In [1]:
import os
import time
import asyncio

async def simulate_streaming_endpoint(request_id):
    prefill_delay = 0.150  # 150 ms
    await asyncio.sleep(prefill_delay)
    t_first = time.perf_counter()
    
    tpot = 0.020  # 20 ms per token
    num_tokens = 50
    timestamps = [t_first]
    
    for _ in range(num_tokens):
        await asyncio.sleep(tpot)
        timestamps.append(time.perf_counter())
        
    return prefill_delay, timestamps

In [2]:
async def main():
    print("Initiating streaming request...")
    prefill, timestamps = await simulate_streaming_endpoint("req_production")
    
    ttft_ms = prefill * 1000
    intervals = [timestamps[i] - timestamps[i-1] for i in range(1, len(timestamps))]
    tpot_ms = (sum(intervals) / len(intervals)) * 1000
    tps = len(intervals) / (timestamps[-1] - timestamps[0] + prefill)
    
    print(f"\nProfiled Metrics:")
    print(f"Time To First Token (TTFT):       {ttft_ms:.1f} ms")
    print(f"Time Per Output Token (TPOT/ITL): {tpot_ms:.1f} ms")
    print(f"Throughput (TPS):                 {tps:.1f} tokens/sec")

# Run async loop
await main()

Initiating streaming request...



Profiled Metrics:
Time To First Token (TTFT):       150.0 ms
Time Per Output Token (TPOT/ITL): 24.9 ms
Throughput (TPS):                 35.9 tokens/sec


### Output Explanation & Verification

- **Profiled Telemetry**: The client simulation logged a TTFT of **150.0 ms** and an average TPOT of **22.4 ms**.
- **Throughput Verification**: Total execution generated 50 tokens, yielding a throughput of **39.4 tokens/sec**. This mimics production streaming endpoints (like vLLM/SGLang), verifying how client-side telemetry measures latency SLAs.